<a href="https://colab.research.google.com/github/dynamodenis/QDrant/blob/main/sparse_vectors/hybrid_search_RRF_DBSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hybrid Search System
Retrieval in Qdrant

To interact with Qdrant, we'll install the Qdrant Python client and Qdrant's lightweight embedding library called [FastEmbed](https://github.com/qdrant/fastembed).


### Step 1: Install the Qdrant Client & FastEmbed

In [ ]:
!pip install -q "qdrant-client[fastembed]>=1.14.2"
!pip3 install -U -q fastembed

## Step 2: Import Required Libraries

In [ ]:
from qdrant_client import QdrantClient, models

## Step 3: Connect to Qdrant Cloud

In [ ]:
from google.colab import userdata

client = QdrantClient(
    url=userdata.get("QDRANT_URL"),
    api_key=userdata.get('QDRANT_API_KEY')
)

## Create a COllection with Named Vectors

For hybrid search, we need a collection that supports both sparse and dense vectors. Qdrant allows multiple named vectors per point:

In [ ]:
from qdrant_client import models

# Define the collection name
collection_name = "hybrid_search_demo"

# Create our collection with both sparse (bm25) and dense vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
            distance=models.Distance.COSINE,
            size=384,
        ),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)


Key configuration details:

    Named vectors: "dense" and "sparse" identify each vector type
    Dense configuration: 384 dimensions (matches sentence-transformers/all-MiniLM-L6-v2)
    Cosine distance: Typical choice for semantic similarity
    IDF modifier: Inverse Document Frequency weighting for BM25 sparse vectors
    Same collection: Both vectors exist on the same points for hybrid search

## Upload the Cheese Dataset

We’re using a small dataset of 10 documents describing different types of cheese and cheese-based dishes. This simple dataset makes it easy to observe the behavior of different search methods:

In [ ]:
documents = [
    "Aged Gouda develops a crystalline texture and nutty flavor profile after 18 months of maturation.",
    "Mature Gouda cheese becomes grainy and develops a rich, buttery taste with extended aging.",
    "Brie cheese features a soft, creamy interior surrounded by an edible white rind.",
    "This French cheese has a flowing, buttery center encased in a bloomy white crust.",
    "Fresh mozzarella pairs beautifully with ripe tomatoes and basil leaves.",
    "Classic Margherita pizza topped with tomato sauce, mozzarella, and fresh basil.",
    "Parmesan requires at least 12 months of cave aging to develop its signature sharp taste.",
    "Parmigiano-Reggiano's distinctive piquant flavor comes from extended maturation in controlled environments.",
    "Grilled cheese sandwiches are the ultimate American comfort food for cold winter days.",
    "Croque Monsieur combines ham and Gruyère in France's answer to the toasted cheese sandwich.",
]

import uuid

client.upsert(
    collection_name=collection_name,
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "dense": models.Document(
                    text=doc,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                "sparse": models.Document(
                    text=doc,
                    model="Qdrant/bm25",
                ),
            },
            payload={"text": doc},
        )
        for doc in documents
    ]
)



About this approach:

 1. Document model: Automatically generates embeddings using specified models
 2. Dual embedding: Each point gets both dense and sparse representations
  3.  Small dataset: 10 documents is perfect for observing search behavior differences
  4. No batching needed: For production with larger datasets, implement batching and retry logic

# Compare Dense vs. Sparse Search

In [ ]:
# Dense search
def dense_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="sentence-transformers/all-MiniLM-L6-v2",
        ),
        using="dense",
        limit=3,
    )
    return response.points

# Sparse search
def sparse_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="sparse",
        limit=3,
    )
    return response.points


# Test Queries Across Both Methods

Now let’s run both methods on different query types:

In [ ]:
queries = [
    "nutty aged cheese",
    "soft French cheese",
    "pizza ingredients",
    "a good lunch",
]

for query in queries:
    print("Query:", query)

    dense_results = dense_search(query)
    print("Dense Results:")
    for result in dense_results:
        print("\t-", result.payload["text"], result.score)

    sparse_results = sparse_search(query)
    print("Sparse Results:")
    for result in sparse_results:
        print("\t-", result.payload["text"], result.score)
    print()


## Key Observations from Results

    1. Dense and sparse produce different rankings: For “nutty aged cheese”, sparse correctly identifies the exact match as #1, while dense ranks a semantically similar document higher
    2. Sometimes rankings match: For “soft French cheese”, both methods agree on the top results, but with different confidence scores
    3. Dense always returns expected results: Dense search will always return 3 results because any two vectors have some similarity, even if extremely low
    4. Sparse can return fewer results: For “pizza ingredients”, sparse only returns 1 result. For “a good lunch”, sparse returns 0 results due to vocabulary mismatch
    5. Vocabulary mismatch problem: When query terms don’t appear in documents, sparse search fails completely, while dense understands the semantic intent

# Hybrid Search with Reciprocal Rank Fusion

Now let’s combine both methods using RRF. This fusion algorithm doesn’t compare incompatible scores - it only uses the ranking order:

In [ ]:
def rrf_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    return response.points


## RRF Results Analysis

Result: Documents that perform well in both methods rank higher

In [ ]:
for query in queries:
    print("Query:", query)

    rrf_results = rrf_search(query)
    print("RRF Results:")
    for result in rrf_results:
        print("\t-", result.payload["text"], result.score)
    print()


## What to notice:

    1. Best of both worlds: Results include documents from both dense and sparse searches
    2. Ranking preservation: When both methods agree (like “soft French cheese”), RRF maintains the consensus
    3. Handles sparse gaps: When sparse returns fewer results (or none), dense search fills the gaps
    5. Balanced scoring: Documents ranked highly by both methods get boosted in RRF scores

# Distribution-Based Score Fusion (DBSF)

RRF isn’t the only fusion method available. DBSF normalizes scores from each query and sums them across different retrievers:

In [ ]:
def dbsf_search(query: str) -> list[models.ScoredPoint]:
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.DBSF),
        limit=3,
    )
    return response.points


## DBSF Results

In [ ]:
for query in queries:
    print("Query:", query)

    dbsf_results = dbsf_search(query)
    print("DBSF Results:")
    for result in dbsf_results:
        print("\t-", result.payload["text"], result.score)
    print()


## Comparing DBSF to RRF:

    1. In this simple example, DBSF and RRF produce identical rankings for all queries
    2. This is NOT a general rule - different fusion methods can produce different results
    3. With larger datasets and more complex queries, the differences become more apparent
    4. DBSF considers score distributions, while RRF only uses rank positions

# Production recommendations:

    1. Start with RRF - it’s simple and effective
    2. Test DBSF if you need score-distribution awareness
    3. Build evaluation datasets for your specific domain
    4. Monitor user satisfaction metrics
    5. Consider adding specialized rerankers for even better quality